## 🎯 Learning Objectives
* Understand the concept of adaptive retrieval and query transformation in advanced RAG systems.
* Learn to implement various query transformation techniques (e.g., expansion, rephrasing, decomposition) using LangChain and LangGraph.
* Evaluate the impact of query transformation on retrieval performance, relevance, and user experience.
* Identify appropriate scenarios and trade-offs for deploying adaptive query transformation in production-grade RAG applications.


## Adaptive Retrieval with Query Transformation: Enhancing RAG Precision

In the evolving landscape of Retrieval Augmented Generation (RAG) systems, a common challenge is the 'impedance mismatch' between a user's natural language query and the optimal query for a retrieval system. A user might ask a broad, ambiguous, or multi-faceted question, but a simple keyword search or vector similarity search might only capture one aspect, leading to suboptimal retrieval and, consequently, less accurate generations.

This is where **Adaptive Retrieval with Query Transformation** comes into play. Imagine a seasoned detective who, upon receiving a vague lead, doesn't just follow it blindly. Instead, they might:

1.  **Rephrase the lead:** "Who was seen near the warehouse?" becomes "Can you describe individuals observed in the vicinity of the warehouse between 10 PM and midnight?"
2.  **Expand the lead:** "Look for the suspect" becomes "Look for John Doe, or anyone matching his description: tall, dark hair, scar on left cheek."
3.  **Decompose the lead:** "What's the history of the company and its current market position?" becomes two separate inquiries: "What is the founding history and key milestones of the company?" and "What is the company's current market share and competitive landscape?"
4.  **Generate a hypothetical scenario:** Based on initial clues, the detective might hypothesize a sequence of events and then search for evidence that supports or refutes that hypothesis.

Adaptive retrieval applies this same intelligent, dynamic approach to RAG. Instead of a static retrieval process, the system actively analyzes the initial query, potentially assesses initial retrieval results, and then *transforms* the query to improve the chances of finding relevant information. This transformation is often orchestrated by a Large Language Model (LLM) acting as an 'agent' within the RAG pipeline.

### Key Query Transformation Techniques:

*   **Query Expansion:** Adding synonyms, related terms, or breaking down complex queries into sub-queries to broaden the search scope and improve recall.
*   **Query Rephrasing/Reformulation:** Rewriting the query to be more precise, cover different facets, or match the language style and terminology prevalent in the retrieved documents. This can be crucial for bridging the lexical gap.
*   **Query Decomposition:** Breaking a multi-faceted or compound question into simpler, atomic questions. Each sub-question can then be answered independently, and the results synthesized.
*   **Hypothetical Document Generation (HyDE):** The LLM generates a plausible, hypothetical answer to the query. This hypothetical answer is then embedded, and its embedding is used as the query vector for retrieval. This helps align the query's semantic meaning with the document space, even if the original query uses different phrasing.

### Why "Agentic"?

The "agentic" aspect arises because the LLM isn't just performing a fixed transformation. It's often involved in a decision-making loop: *"Is the current query good enough? If not, how should I transform it? Which transformation strategy is best here? Should I try again?"* This involves reasoning, planning, and potentially iterative refinement, making the RAG system more robust and intelligent.

**LangGraph's Role:** LangGraph is an ideal framework for orchestrating these adaptive retrieval workflows. Its stateful, cyclic graph capabilities allow us to define complex decision paths: query -> retrieve -> evaluate -> (if needed) transform query -> retrieve again -> evaluate -> synthesize. This enables the RAG system to dynamically adapt its strategy based on the quality of intermediate results, leading to significantly improved answer quality for challenging queries.


In [ ]:
import os
from typing import List, Dict, Any

# Ensure you have the necessary packages installed:
# pip install langchain langchain-community langchain-openai langgraph chromadb tiktoken

# Set up environment variables for API keys
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGCHAIN_API_KEY"

# --- 1. Initialize LLM and Embeddings ---
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser

# Using a powerful model for complex reasoning tasks in 2026
llm = ChatOpenAI(model="gpt-4o-2024-05-13", temperature=0.1)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# --- 2. Create a Dummy Vector Store (In-memory for demonstration) ---
# In a real-world scenario, this would be a persistent, large-scale vector database.
docs = [
    Document(page_content="Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers.", metadata={"source": "wiki"}),
    Document(page_content="One of the main challenges in quantum computing is decoherence, where quantum states lose their coherence due to interaction with the environment.", metadata={"source": "research_paper"}),
    Document(page_content="Error correction in quantum computers is vital but extremely difficult, requiring many physical qubits to encode a single logical qubit.", metadata={"source": "textbook"}),
    Document(page_content="Superconducting qubits, trapped ions, and topological qubits are leading candidates for building quantum computers, each with unique advantages and challenges.", metadata={"source": "review_article"}),
    Document(page_content="Overcoming decoherence involves isolating qubits from environmental noise and developing robust error correction codes.", metadata={"source": "research_paper"}),
    Document(page_content="The development of fault-tolerant quantum computers is a long-term goal, requiring significant advancements in hardware and software.", metadata={"source": "roadmap"}),
    Document(page_content="Quantum algorithms like Shor's algorithm for factoring and Grover's algorithm for search offer exponential speedups for specific problems.", metadata={"source": "wiki"})
]

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- 3. Define LangGraph Components ---
from langgraph.graph import StateGraph, END

# Define the state for our graph
class GraphState(Dict): # Using Dict for simplicity, can be Pydantic model
    question: str
    transformed_question: str = None
    retrieved_documents: List[Document] = []
    retrieval_score: str = "unscored" # 'good', 'bad'
    generation: str = None
    steps: List[str] = []

# --- Nodes for the LangGraph --- 

# Node 1: Retrieve documents
def retrieve(state: GraphState) -> GraphState:
    print("---RETRIEVING DOCUMENTS---")
    question = state["transformed_question"] or state["question"]
    documents = retriever.invoke(question)
    return {"retrieved_documents": documents, "steps": state["steps"] + [f"Retrieved for: {question}"]}

# Node 2: Grade Retrieval Quality
def grade_retrieval(state: GraphState) -> GraphState:
    print("---GRADING RETRIEVAL---")
    question = state["question"]
    documents = state["retrieved_documents"]

    # LLM-based grading prompt
    grade_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a RAG assistant. Your task is to assess the relevance of retrieved documents to the user's question. "
                     "Respond with 'good' if the documents are highly relevant and sufficient to answer the question, 'bad' otherwise."
                     "Do not provide explanations, just 'good' or 'bad'."),
        ("human", "Question: {question}\n\nDocuments:\n{documents}\n\nAre these documents good enough?")
    ])
    
    grader = grade_prompt | llm | StrOutputParser()
    
    # Concatenate document content for grading
    doc_contents = "\n\n".join([doc.page_content for doc in documents])
    retrieval_score = grader.invoke({"question": question, "documents": doc_contents})
    
    print(f"Retrieval Score: {retrieval_score}")
    return {"retrieval_score": retrieval_score.strip().lower(), "steps": state["steps"] + [f"Graded retrieval as: {retrieval_score}"]}

# Node 3: Transform Query (e.g., decompose or rephrase)
def transform_query(state: GraphState) -> GraphState:
    print("---TRANSFORMING QUERY---")
    original_question = state["question"]
    
    # LLM-based query transformation prompt
    transform_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert query transformer. Given a user's question, if it's complex or ambiguous, "
                     "rephrase it into 2-3 more specific, atomic questions or expand it with relevant keywords to improve retrieval. "
                     "If the question is already simple, just return the original question. "
                     "Return the transformed query/queries as a comma-separated list."),
        ("human", "Original Question: {question}")
    ])
    
    transformer = transform_prompt | llm | StrOutputParser()
    
    transformed_q = transformer.invoke({"question": original_question})
    print(f"Transformed Query: {transformed_q}")
    
    # For simplicity, we'll just use the first transformed query if multiple are returned
    # In a real system, you might retrieve for all and combine results.
    return {"transformed_question": transformed_q.split(',')[0].strip(), "steps": state["steps"] + [f"Transformed query to: {transformed_q.split(',')[0].strip()}"]}

# Node 4: Generate Answer
def generate_answer(state: GraphState) -> GraphState:
    print("---GENERATING ANSWER---")
    question = state["question"]
    documents = state["retrieved_documents"]
    
    generation_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Based on the provided context, answer the user's question comprehensively and accurately. "
                     "If the context does not contain enough information, state that."),
        ("human", "Question: {question}\n\nContext:\n{context}")
    ])
    
    generator = generation_prompt | llm | StrOutputParser()
    
    doc_contents = "\n\n".join([doc.page_content for doc in documents])
    answer = generator.invoke({"question": question, "context": doc_contents})
    
    return {"generation": answer, "steps": state["steps"] + ["Generated answer"]}

# --- Build the LangGraph --- 
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_retrieval", grade_retrieval)
workflow.add_node("transform_query", transform_query)
workflow.add_node("generate_answer", generate_answer)

# Set entry point
workflow.set_entry_point("retrieve")

# Add edges
workflow.add_edge("retrieve", "grade_retrieval")
workflow.add_edge("transform_query", "retrieve") # If query is transformed, retrieve again
workflow.add_edge("generate_answer", END)

# Add conditional edge from grade_retrieval
def decide_to_transform(state: GraphState) -> str:
    print("---DECIDING NEXT STEP---")
    if state["retrieval_score"] == "good":
        print("Decision: Retrieval is good, proceed to generate answer.")
        return "generate_answer"
    else:
        print("Decision: Retrieval is bad, transform query and retry.")
        return "transform_query"

workflow.add_conditional_edges(
    "grade_retrieval",
    decide_to_transform,
    {
        "generate_answer": "generate_answer",
        "transform_query": "transform_query"
    }
)

# Compile the graph
app = workflow.compile()

# --- Run the Adaptive RAG System ---
print("\n--- Running Adaptive RAG with Query Transformation ---")

# Example 1: A complex query that might benefit from transformation
complex_query = "What are the primary hurdles in quantum computing and how can we overcome them?"
print(f"\nInitial Query: {complex_query}")

# The initial state for the graph
initial_state = {"question": complex_query, "steps": []}

# Invoke the graph
final_state = app.invoke(initial_state)

print("\n--- Final Result (Complex Query) ---")
print(f"Question: {final_state['question']}")
print(f"Transformed Question: {final_state['transformed_question']}")
print(f"Retrieval Score: {final_state['retrieval_score']}")
print(f"Generated Answer: {final_state['generation']}")
print(f"Steps taken: {final_state['steps']}")

print("\n" + "="*50 + "\n")

# Example 2: A simpler query that might not need transformation
simple_query = "What is Shor's algorithm?"
print(f"\nInitial Query: {simple_query}")

initial_state_simple = {"question": simple_query, "steps": []}
final_state_simple = app.invoke(initial_state_simple)

print("\n--- Final Result (Simple Query) ---")
print(f"Question: {final_state_simple['question']}")
print(f"Transformed Question: {final_state_simple['transformed_question']}")
print(f"Retrieval Score: {final_state_simple['retrieval_score']}")
print(f"Generated Answer: {final_state_simple['generation']}")
print(f"Steps taken: {final_state_simple['steps']}")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a basic yet powerful adaptive RAG system using LangGraph. Let's break down its execution and implications:

1.  **Initial Retrieval:** The graph starts by attempting retrieval with the original user query. This is the baseline. The `retrieve` node fetches documents from our dummy vector store.
2.  **Retrieval Grading:** The `grade_retrieval` node, powered by an LLM, acts as a critic. It assesses whether the retrieved documents are relevant and sufficient to answer the *original* question. This is a crucial 'agentic' step, as it introduces a feedback loop.
3.  **Conditional Logic (Adaptation):** Based on the `retrieval_score`:
    *   If the score is 'good', the system proceeds directly to `generate_answer`. This avoids unnecessary computation for simple, well-posed queries.
    *   If the score is 'bad', the system enters the adaptive loop, moving to `transform_query`.
4.  **Query Transformation:** The `transform_query` node, another LLM-powered agent, takes the original question and attempts to rephrase, expand, or decompose it into a more effective query for retrieval. In our example, for a complex query like "What are the primary hurdles in quantum computing and how can we overcome them?", the LLM might transform it into something like "main challenges in quantum computing, overcoming decoherence, quantum error correction."
5.  **Re-retrieval:** With the transformed query, the system re-enters the `retrieve` node, aiming to fetch more relevant documents.
6.  **Answer Generation:** Finally, the `generate_answer` node synthesizes the answer using the (potentially improved) retrieved documents.

**Example 1 (Complex Query):** You'll likely observe that for the complex query, the `grade_retrieval` node initially returns 'bad'. This triggers the `transform_query` node, which generates a more specific query. The subsequent retrieval with this transformed query should yield more relevant documents, leading to a better final answer. The `steps` list in the final state clearly illustrates this adaptive path.

**Example 2 (Simple Query):** For a straightforward query like "What is Shor's algorithm?", the initial retrieval might be deemed 'good' by the grader, and the system will bypass the transformation step, directly generating an answer. This showcases the *adaptive* nature – only transforming when necessary.

### Performance Trade-offs:

While adaptive retrieval significantly enhances answer quality for complex queries, it introduces several trade-offs:

*   **Increased Latency:** Each LLM call (for grading, transformation, and generation) and each retrieval step adds to the overall response time. For real-time applications, this needs careful optimization.
*   **Higher Cost:** More LLM interactions directly translate to higher API costs. The benefit of improved accuracy must outweigh the increased operational expense.
*   **System Complexity:** Adding conditional logic and multiple LLM agents makes the RAG pipeline more intricate to design, debug, and maintain. Monitoring the performance of each agent (grader, transformer) becomes critical.
*   **Potential for Hallucination/Misdirection:** If the LLM-based grader or transformer makes a poor decision, it can lead the system down an irrelevant path, potentially worsening the answer or increasing latency unnecessarily.

### Typical Use Cases:

Adaptive retrieval with query transformation is particularly beneficial in scenarios where:

*   **Complex or Ambiguous Queries:** Users frequently ask multi-part, vague, or highly nuanced questions that a single, direct search struggles with.
*   **High-Stakes Information Retrieval:** Domains like legal research, medical diagnostics, or financial analysis where accuracy and comprehensiveness are paramount.
*   **Conversational AI and Chatbots:** To handle follow-up questions, clarify user intent, and provide more robust answers in multi-turn conversations.
*   **Diverse Document Corpora:** When the knowledge base contains documents with varying terminology, styles, or levels of detail, requiring flexible query strategies.
*   **Enterprise Search:** Improving the discoverability of information across vast and heterogeneous internal knowledge bases.

By intelligently adapting the retrieval strategy, RAG systems can move beyond simple keyword matching to truly understand and fulfill complex information needs, delivering a superior user experience.


### Resources for Further Exploration

*   **LangChain Documentation - LangGraph:** The official documentation for building stateful, multi-actor applications with LLMs. This is your primary resource for advanced RAG orchestration.
    *   [https://python.langchain.com/docs/langgraph](https://python.langchain.com/docs/langgraph)
*   **LangChain Documentation - Retrieval:** Explore various retrieval strategies, including advanced techniques like query expansion and rephrasing.
    *   [https://python.langchain.com/docs/modules/data_connection/retrievers/](https://python.langchain.com/docs/modules/data_connection/retrievers/)
*   **OpenAI API Documentation:** For understanding the capabilities and best practices of models like GPT-4o, which are excellent for query transformation and grading tasks.
    *   [https://platform.openai.com/docs/api-reference](https://platform.openai.com/docs/api-reference)
*   **Google AI Studio / Gemini API:** Explore Google's powerful Gemini models for similar LLM-driven tasks.
    *   [https://ai.google.dev/](https://ai.google.dev/)
*   **Hugging Face Transformers Library:** While not directly used for the LLM calls in this example, Hugging Face is central to the broader AI ecosystem, especially for embedding models and fine-tuning smaller LMs for specific tasks.
    *   [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **Research Papers on Query Transformation/RAG:**
    *   **"Query Rewriting for Retrieval-Augmented Generation"** (e.g., from Google Research or similar venues) for deeper insights into LLM-based query reformulation.
    *   **"HyDE: Hypothetical Document Embeddings for Improved Zero-Shot Retrieval"** for understanding the HyDE technique.
    *   Look for recent papers on "Agentic RAG" or "Adaptive RAG" on arXiv or major NLP conferences (ACL, EMNLP, NeurIPS) for the latest advancements.
